# WEEK BY WEEK WINNER PREDICTIONS

## Potential name for application: Gamelytics

In [1]:
import nfl_data_py as nfl
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from tensorflow import keras
from keras import layers

In [2]:
print(nfl.import_schedules([2025]))

              game_id  season game_type  week     gameday   weekday gametime  \
6991  2025_01_DAL_PHI    2025       REG     1  2025-09-04  Thursday    20:20   
6992   2025_01_KC_LAC    2025       REG     1  2025-09-05    Friday    20:00   
6993   2025_01_TB_ATL    2025       REG     1  2025-09-07    Sunday    13:00   
6994  2025_01_CIN_CLE    2025       REG     1  2025-09-07    Sunday    13:00   
6995  2025_01_MIA_IND    2025       REG     1  2025-09-07    Sunday    13:00   
...               ...     ...       ...   ...         ...       ...      ...   
7258  2025_18_DAL_NYG    2025       REG    18  2026-01-04    Sunday    13:00   
7259  2025_18_WAS_PHI    2025       REG    18  2026-01-04    Sunday    13:00   
7260  2025_18_BAL_PIT    2025       REG    18  2026-01-04    Sunday    13:00   
7261   2025_18_SEA_SF    2025       REG    18  2026-01-04    Sunday    13:00   
7262   2025_18_CAR_TB    2025       REG    18  2026-01-04    Sunday    13:00   

     away_team  away_score home_team  .

## 1. Load the schedule data

In [3]:
train_seasons = list(range(2020, 2025))

predict_season = 2025

sched = nfl.import_schedules(train_seasons)

sched_pred = nfl.import_schedules([predict_season])

## 2. Compute team/season features

In [4]:
team_stats = []

for season in train_seasons:
    # get games from this season
    season_games = sched[sched['season'] == season]

    # get all teams that played this season
    all_teams = list(season_games['home_team'].unique()) + list(season_games['away_team'].unique())
    all_teams = list(set(all_teams))  # remove duplicates

    for team in all_teams:
        # games where this team was home or away
        home_games = season_games[season_games['home_team'] == team]
        away_games = season_games[season_games['away_team'] == team]

        # total points scored and allowed
        points_for = home_games['home_score'].sum() + away_games['away_score'].sum()
        points_against = home_games['away_score'].sum() + away_games['home_score'].sum()

        # count wins and losses
        home_wins = (home_games['home_score'] > home_games['away_score']).sum()
        away_wins = (away_games['away_score'] > away_games['home_score']).sum()
        wins = int(home_wins + away_wins)

        home_losses = (home_games['home_score'] < home_games['away_score']).sum()
        away_losses = (away_games['away_score'] < away_games['home_score']).sum()
        losses = int(home_losses + away_losses)

        games_played = len(home_games) + len(away_games)

        # save results
        team_stats.append({
            'season': season,
            'team': team,
            'points_for': points_for,
            'points_against': points_against,
            'wins': wins,
            'losses': losses,
            'games_played': games_played
        })

# make into dataframe
team_feats = pd.DataFrame(team_stats)

# averages
team_feats['avg_points_for'] = team_feats['points_for'] / team_feats['games_played'].replace(0, 1)
team_feats['avg_points_against'] = team_feats['points_against'] / team_feats['games_played'].replace(0, 1)

## 3. Build the matchup dataset for training

In [5]:
matchups = []
for _, g in sched.iterrows():
    try:
        home = g['home_team']
        away = g['away_team']
        season = g['season']
        week = g['week'] if 'week' in g else np.nan

        hf = team_feats[(team_feats['team']==home) & (team_feats['season']==season)].iloc[0]
        af = team_feats[(team_feats['team']==away) & (team_feats['season']==season)].iloc[0]

        matchups.append({
            'season': season,
            'week': week,
            'matchup': f"{away} @ {home}",
            'home_team': home,
            'away_team': away,
            'points_for_diff': hf['avg_points_for'] - af['avg_points_for'],
            'points_against_diff': hf['avg_points_against'] - af['avg_points_against'],
            'wins_diff': hf['wins'] - af['wins'],
            'label': 1 if g['home_score'] > g['away_score'] else 0
        })
    except:
        continue

matchups_df = pd.DataFrame(matchups)

feature_cols = ['points_for_diff','points_against_diff','wins_diff']
X = matchups_df[feature_cols].values
y = matchups_df['label'].values

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Train Models

In [6]:
lr = LogisticRegression(max_iter=2000)
lr.fit(X_tr, y_tr)

# Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_tr, y_tr)

DecisionTreeClassifier(random_state=42)

In [7]:
# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_tr, y_tr)

RandomForestClassifier(n_estimators=200, random_state=42)

In [8]:
# XGBoost
xgbc = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgbc.fit(X_tr, y_tr)

/opt/anaconda3/envs/tensorflowEnv/lib/python3.8/site-packages/xgboost/core.py:158: UserWarning: [17:17:36] WARNING: /var/folders/k1/30mswbxs7r1g6zwn8y4fyt500000gp/T/abs_d9k8pmaj4_/croot/xgboost-split_1724073758172/work/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [9]:
# Neural Network
def make_nn(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

nn = make_nn(X_tr.shape[1])
nn.fit(X_tr, y_tr, epochs=25, batch_size=64, validation_data=(X_val, y_val), verbose=0)

2025-09-24 17:17:37.267649: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


## 5. Ensemble Prediction

In [10]:
models = {'Logistic': lr, 'DecisionTree': dt, 'RandomForest': rf, 'XGBoost': xgbc, 'NeuralNet': nn}
val_scores = {k: (v.score(X_val,y_val) if k != 'NeuralNet' else v.evaluate(X_val,y_val,verbose=0)[1]) for k,v in models.items()}

def ensemble_predict(X_input):
    preds = []
    for name, mdl in models.items():
        if name == 'NeuralNet':
            p = mdl.predict(X_input).flatten()
        else:
            p = mdl.predict_proba(X_input)[:,1]
        preds.append(p)
    preds = np.vstack(preds)
    accs = np.array([val_scores[nm] for nm in models.keys()])
    weights = accs / accs.sum()
    weighted = np.dot(weights, preds)
    return weighted

## 6. Predict 2025 matchups

In [11]:
current_week = 4  # change this to the actual current week

# Build features for prediction season using 2024 team feats
feat_2024 = team_feats[team_feats['season']==2024].copy()
pred_rows = []
for _, g in sched_pred.iterrows():
    try:
        if g['week'] != current_week:
            continue  # skip other weeks
        home = g['home_team']
        away = g['away_team']
        hf = feat_2024[feat_2024['team']==home].iloc[0]
        af = feat_2024[feat_2024['team']==away].iloc[0]

        pred_rows.append({
            'matchup': f"{away} @ {home}",
            'home_team': home,
            'away_team': away,
            'points_for_diff': hf['avg_points_for'] - af['avg_points_for'],
            'points_against_diff': hf['avg_points_against'] - af['avg_points_against'],
            'wins_diff': hf['wins'] - af['wins']
        })
    except:
        continue

pred_df = pd.DataFrame(pred_rows)
X_pred = pred_df[feature_cols].values
weighted_probs = ensemble_predict(X_pred)

# Predicted winner
pred_df['predicted_winner'] = np.where(weighted_probs >= 0.5, pred_df['home_team'], pred_df['away_team'])

# Confidence as percentage
pred_df['confidence'] = np.where(
    pred_df['predicted_winner'] == pred_df['home_team'],
    weighted_probs * 100,
    (1 - weighted_probs) * 100
)

# Round for cleaner display
pred_df['confidence'] = pred_df['confidence'].round(1)

1/1 [==============================] - 0s 72ms/step


## 7. Output current weeks predictions

In [12]:
current_week_out = pred_df[['matchup','predicted_winner','confidence']]
current_week_out


,matchup,predicted_winner,confidence
0,SEA @ ARI,SEA,77.9
1,MIN @ PIT,PIT,51.2
2,WAS @ ATL,WAS,60.1
3,NO @ BUF,BUF,90.0
4,CLE @ DET,DET,98.8
5,TEN @ HOU,HOU,56.3
6,CAR @ NE,CAR,52.6
7,LAC @ NYG,LAC,85.6
8,PHI @ TB,PHI,88.0
9,IND @ LA,LA,79.5
